# 04 — The two-tower model: context, cold-start, and an ablation

This is the centerpiece notebook. It builds two models on the exact same data
and evaluation protocol:

- **`plain`** — a bare two-tower model: just user id and item id/genres. This
  is roughly what a matrix-factorization-era system looks like in two-tower
  form.
- **`enriched`** — adds everything built in notebooks 2 & 3: request-time
  context (day/time-of-day), the "user-take" recent-history feature (with the
  cold-start training trick — dropping the real user id for 15% of rows), and
  frozen item text embeddings.

Then it measures, with real numbers, whether `enriched` actually earns its
complexity — overall, and specifically for simulated cold-start users. That
comparison is the empirical answer to ADR-001, and the basis for ADR-004.

As noted in ADR-007: this is a from-scratch PyTorch implementation, not a
call into `merlin.models.tf.TwoTowerModel` — the real Merlin library runs in
`training/modal_app.py` on a Modal GPU, inside NVIDIA's own container, where
its 2023-era pinned dependencies don't conflict with anything. This notebook
is where I actually learn how the architecture works.


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

torch.manual_seed(0)
np.random.seed(0)

PROCESSED_DIR = Path("..") / "data" / "processed"
ARTIFACTS_DIR = Path("..") / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

train_df = pd.read_parquet(PROCESSED_DIR / "train.parquet")
val_df = pd.read_parquet(PROCESSED_DIR / "val.parquet")
test_df = pd.read_parquet(PROCESSED_DIR / "test.parquet")
movie_genres = pd.read_parquet(PROCESSED_DIR / "movie_genres.parquet")

with open(PROCESSED_DIR / "vocab.json") as f:
    vocab = json.load(f)

movie_vocab = vocab["movie_vocab"]            # movieId(str) -> movie_idx
genre_vocab = vocab["genre_vocab"]
day_vocab = vocab["day_vocab"]
tod_vocab = vocab["tod_vocab"]
K = vocab["user_take_k"]

n_users = len(vocab["user_vocab"]) + 1
n_movies = len(movie_vocab) + 1
n_genres = len(genre_vocab) + 1
n_days = len(day_vocab) + 1
n_tods = len(tod_vocab) + 1

text_npz = np.load(PROCESSED_DIR / "item_text_embeddings.npz")
text_movie_ids = text_npz["movie_ids"]
text_embs_raw = text_npz["embeddings"]
TEXT_DIM = text_embs_raw.shape[1]

len(train_df), len(val_df), len(test_df), n_users, n_movies, TEXT_DIM


(99616, 610, 610, 611, 9725, 384)

## Aligning everything to `movie_idx`

Notebook 2/3 produced arrays keyed by `movieId`. The model only knows
`movie_idx` (the Categorify output). So the first job here is building
`movie_idx`-indexed lookup tables for genres and text embeddings — row 0 is
the reserved UNK/pad slot and gets a zero vector.


In [2]:
MAX_GENRES = 6

genre_idx_by_movie_idx = np.zeros((n_movies, MAX_GENRES), dtype=np.int64)
for _, row in movie_genres.iterrows():
    midx = movie_vocab.get(str(row["movieId"]))
    if midx is None:
        continue
    g = row["genre_idxs"][:MAX_GENRES]
    genre_idx_by_movie_idx[midx, : len(g)] = g

text_emb_by_movie_idx = np.zeros((n_movies, TEXT_DIM), dtype=np.float32)
for mid, emb in zip(text_movie_ids, text_embs_raw):
    midx = movie_vocab.get(str(mid))
    if midx is not None:
        text_emb_by_movie_idx[midx] = emb

genre_idx_by_movie_idx.shape, text_emb_by_movie_idx.shape


((9725, 6), (9725, 384))

## Dataset + masked mean pooling

In [3]:
class InteractionsDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.user_idx = df["user_idx_for_training"].to_numpy()
        self.movie_idx = df["movie_idx"].to_numpy()
        self.day_idx = df["day_idx"].to_numpy()
        self.tod_idx = df["tod_idx"].to_numpy()
        self.user_take = np.stack(df["user_take"].to_numpy())

    def __len__(self):
        return len(self.movie_idx)

    def __getitem__(self, i):
        return {
            "user_idx": self.user_idx[i],
            "movie_idx": self.movie_idx[i],
            "day_idx": self.day_idx[i],
            "tod_idx": self.tod_idx[i],
            "user_take": self.user_take[i],
        }


def masked_mean(embedded: torch.Tensor, idxs: torch.Tensor) -> torch.Tensor:
    """embedded: (B, L, D) ; idxs: (B, L) ; ignores positions where idxs == 0."""
    mask = (idxs != 0).unsqueeze(-1).float()
    summed = (embedded * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1.0)
    return summed / counts


train_ds = InteractionsDataset(train_df)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, drop_last=True)
len(train_ds)


99616

`shuffle=True` here isn't just a DataLoader default — it's load-bearing. The
training loss below uses **in-batch negatives**: every other item in the same
batch stands in as a negative example for a given user. If the data weren't
shuffled (e.g. sorted by `userId`), a batch would be mostly one user's own
positive items, which would make for trivially easy, useless negatives.


## The model: two towers, built from the same pieces

In [4]:
EMB_USER, EMB_MOVIE, EMB_GENRE, EMB_DAY, EMB_TOD = 16, 16, 8, 4, 4
TEXT_PROJ_DIM = 16
TOWER_OUT_DIM = 32


class SharedEmbeddings(nn.Module):
    """Embedding tables shared between the query tower (for user-take) and
    the item tower (for the item id itself) — one movie, one vector."""

    def __init__(self):
        super().__init__()
        self.movie = nn.Embedding(n_movies, EMB_MOVIE, padding_idx=0)
        self.genre = nn.Embedding(n_genres, EMB_GENRE, padding_idx=0)


class ItemTower(nn.Module):
    def __init__(self, shared: SharedEmbeddings, use_text: bool):
        super().__init__()
        self.shared = shared
        self.use_text = use_text
        in_dim = EMB_MOVIE + EMB_GENRE
        if use_text:
            self.text_proj = nn.Linear(TEXT_DIM, TEXT_PROJ_DIM)
            in_dim += TEXT_PROJ_DIM
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(), nn.Linear(64, TOWER_OUT_DIM)
        )

    def forward(self, movie_idx, genre_idx, text_emb=None):
        m = self.shared.movie(movie_idx)
        g = masked_mean(self.shared.genre(genre_idx), genre_idx)
        parts = [m, g]
        if self.use_text:
            parts.append(self.text_proj(text_emb))  # text_emb is frozen input, proj is trainable
        x = torch.cat(parts, dim=-1)
        out = self.mlp(x)
        return nn.functional.normalize(out, dim=-1)


class QueryTower(nn.Module):
    def __init__(self, shared: SharedEmbeddings, use_context: bool):
        super().__init__()
        self.shared = shared
        self.use_context = use_context
        self.user = nn.Embedding(n_users, EMB_USER, padding_idx=0)
        in_dim = EMB_USER
        if use_context:
            self.day = nn.Embedding(n_days, EMB_DAY, padding_idx=0)
            self.tod = nn.Embedding(n_tods, EMB_TOD, padding_idx=0)
            in_dim += EMB_MOVIE + EMB_DAY + EMB_TOD  # + user_take pooled
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(), nn.Linear(64, TOWER_OUT_DIM)
        )

    def forward(self, user_idx, day_idx=None, tod_idx=None, user_take=None):
        u = self.user(user_idx)
        parts = [u]
        if self.use_context:
            take_pooled = masked_mean(self.shared.movie(user_take), user_take)
            parts += [take_pooled, self.day(day_idx), self.tod(tod_idx)]
        x = torch.cat(parts, dim=-1)
        out = self.mlp(x)
        return nn.functional.normalize(out, dim=-1)


class TwoTower(nn.Module):
    def __init__(self, enriched: bool):
        super().__init__()
        self.enriched = enriched
        self.shared = SharedEmbeddings()
        self.item_tower = ItemTower(self.shared, use_text=enriched)
        self.query_tower = QueryTower(self.shared, use_context=enriched)

    def forward(self, batch, genre_table, text_table):
        movie_idx = batch["movie_idx"]
        genre_idx = torch.as_tensor(genre_table[movie_idx.numpy()])
        item_kwargs = {}
        if self.enriched:
            item_kwargs["text_emb"] = torch.as_tensor(text_table[movie_idx.numpy()])
        item_emb = self.item_tower(movie_idx, genre_idx, **item_kwargs)

        query_kwargs = {}
        if self.enriched:
            query_kwargs = dict(
                day_idx=batch["day_idx"], tod_idx=batch["tod_idx"], user_take=batch["user_take"]
            )
        user_emb = self.query_tower(batch["user_idx"], **query_kwargs)
        return user_emb, item_emb


Notice `TwoTower.__init__` takes one flag, `enriched`. The `plain` and
`enriched` models in this notebook are literally the same class with
different wiring — which is the point: the comparison below isn't "two
different architectures," it's "how much does each enrichment actually buy,"
measured by turning them off.


## In-batch sampled-softmax training loss

In [5]:
def in_batch_loss(user_emb: torch.Tensor, item_emb: torch.Tensor) -> torch.Tensor:
    logits = user_emb @ item_emb.T  # (B, B); cosine similarity since both are L2-normalized
    labels = torch.arange(logits.shape[0])
    return nn.functional.cross_entropy(logits * 10.0, labels)  # temperature-scaled


def train_model(enriched: bool, epochs: int = 20, lr: float = 2e-3) -> TwoTower:
    model = TwoTower(enriched=enriched)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        total_loss, n_batches = 0.0, 0
        for batch in train_loader:
            batch = {k: torch.as_tensor(v) for k, v in batch.items()}
            user_emb, item_emb = model(batch, genre_idx_by_movie_idx, text_emb_by_movie_idx)
            loss = in_batch_loss(user_emb, item_emb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += loss.item()
            n_batches += 1
        print(f"[{'enriched' if enriched else 'plain':9s}] epoch {epoch + 1}/{epochs}  "
              f"loss={total_loss / n_batches:.4f}")
    model.eval()
    return model


The temperature scale (`logits * 10.0`) matters more than it looks: both
embeddings are unit vectors, so raw cosine similarities sit in a narrow
[-1, 1] range — too narrow for softmax/cross-entropy to produce a confident,
learnable gradient. Scaling up (dividing by a small temperature) is standard
practice for this training objective, not a tuning afterthought.


In [6]:
plain_model = train_model(enriched=False)


[plain    ] epoch 1/20  loss=5.5528


[plain    ] epoch 2/20  loss=5.4966


[plain    ] epoch 3/20  loss=5.4503


[plain    ] epoch 4/20  loss=5.4060


[plain    ] epoch 5/20  loss=5.3463


...

[plain    ] epoch 16/20  loss=4.9404


[plain    ] epoch 17/20  loss=4.9262


[plain    ] epoch 18/20  loss=4.9126


[plain    ] epoch 19/20  loss=4.8998


[plain    ] epoch 20/20  loss=4.8871


In [7]:
enriched_model = train_model(enriched=True)


[enriched ] epoch 1/20  loss=5.5414


[enriched ] epoch 2/20  loss=5.2797


[enriched ] epoch 3/20  loss=5.0495


[enriched ] epoch 4/20  loss=4.8876


[enriched ] epoch 5/20  loss=4.7652


...

[enriched ] epoch 16/20  loss=4.2429


[enriched ] epoch 17/20  loss=4.2179


[enriched ] epoch 18/20  loss=4.1956


[enriched ] epoch 19/20  loss=4.1738


[enriched ] epoch 20/20  loss=4.1535


## Evaluation: Recall@K / NDCG@K, exhaustive over the whole catalog

In [8]:
ALL_MOVIE_IDX = torch.arange(n_movies)


@torch.no_grad()
def all_item_embeddings(model: TwoTower) -> torch.Tensor:
    genre_idx = torch.as_tensor(genre_idx_by_movie_idx)
    kwargs = {}
    if model.enriched:
        kwargs["text_emb"] = torch.as_tensor(text_emb_by_movie_idx)
    return model.item_tower(ALL_MOVIE_IDX, genre_idx, **kwargs)


@torch.no_grad()
def evaluate(model: TwoTower, eval_df: pd.DataFrame, k: int = 10,
             force_unknown_user: bool = False) -> dict:
    item_embs = all_item_embeddings(model)  # (n_movies, D) — exhaustive scoring, small catalog

    # eval rows always carry their real user id (the 15% cold-start dropout
    # mask only ever applies to *training* rows); `force_unknown_user` is how
    # this function simulates a brand-new user instead, on demand.
    user_idx = torch.as_tensor(
        np.zeros(len(eval_df), dtype=np.int64) if force_unknown_user
        else eval_df["user_idx"].to_numpy()
    )

    query_kwargs = {}
    if model.enriched:
        query_kwargs = dict(
            day_idx=torch.as_tensor(eval_df["day_idx"].to_numpy()),
            tod_idx=torch.as_tensor(eval_df["tod_idx"].to_numpy()),
            user_take=torch.as_tensor(np.stack(eval_df["user_take"].to_numpy())),
        )
    user_embs = model.query_tower(user_idx, **query_kwargs)  # (B, D)

    scores = user_embs @ item_embs.T  # (B, n_movies) — every item scored for every eval user
    target = torch.as_tensor(eval_df["movie_idx"].to_numpy())

    topk = scores.topk(k, dim=1).indices  # (B, k)
    hits = (topk == target.unsqueeze(1)).any(dim=1)
    recall_at_k = hits.float().mean().item()

    rank_in_topk = (topk == target.unsqueeze(1)).float().argmax(dim=1)
    ndcg = torch.where(hits, 1.0 / torch.log2(rank_in_topk.float() + 2), torch.zeros_like(hits.float()))
    ndcg_at_k = ndcg.mean().item()

    return {"recall_at_k": recall_at_k, "ndcg_at_k": ndcg_at_k, "n": len(eval_df)}


K_EVAL = 10
results = {
    "plain": evaluate(plain_model, test_df, k=K_EVAL),
    "enriched": evaluate(enriched_model, test_df, k=K_EVAL),
    "enriched_cold_start": evaluate(enriched_model, test_df, k=K_EVAL, force_unknown_user=True),
}
pd.DataFrame(results).T


,recall_at_k,ndcg_at_k,n
plain,0.009836,0.005229,610.0
enriched,0.014754,0.005470,610.0
enriched_cold_start,0.021311,0.010718,610.0


`enriched_cold_start` re-scores the *same* test rows but forces the user
embedding to come entirely from `UNK` + that row's real `user_take` history
and context — simulating what a brand-new, never-seen-before user would get.
It only works at all because 15% of training rows had their real user id
dropped the same way (notebook 2, step 5) — without that training-time trick,
an untrained `UNK` embedding would produce close to random recommendations.

**Read the actual printed numbers above before writing ADR-001 and ADR-004** —
that's the whole point of doing the ablation instead of asserting the
conclusion.


## Wait — is that cold-start number real, or noise?

The first run of this notebook showed `enriched_cold_start` beating
`enriched` on Recall@10 — recommendations get *better* when the real,
trained user-id embedding is thrown away and replaced with `UNK`. That's
surprising enough that it's worth checking before writing it into an ADR: is
it a reproducible effect, or is it just noise from evaluating on only 610
held-out examples with a ~1-3% hit rate (where the standard error on Recall@K
is comparable in size to the differences I'm looking at)?

I re-ran training with three different random seeds (same data, same
architecture, same 20 epochs) to check.


In [9]:
def quick_eval_at_seed(seed: int) -> dict:
    torch.manual_seed(seed)
    np.random.seed(seed)
    m = train_model(enriched=True, epochs=20)
    return {
        "seed": seed,
        "enriched": evaluate(m, test_df, k=K_EVAL)["recall_at_k"],
        "enriched_cold_start": evaluate(m, test_df, k=K_EVAL, force_unknown_user=True)["recall_at_k"],
    }


seed_results = [quick_eval_at_seed(s) for s in (1, 7, 42)]
pd.DataFrame(seed_results)


[enriched ] epoch 1/20  loss=5.5169


[enriched ] epoch 2/20  loss=5.2443


[enriched ] epoch 3/20  loss=5.0435


[enriched ] epoch 4/20  loss=4.9036


[enriched ] epoch 5/20  loss=4.7938


...

[enriched ] epoch 17/20  loss=4.2125


[enriched ] epoch 18/20  loss=4.1898


[enriched ] epoch 19/20  loss=4.1665


[enriched ] epoch 20/20  loss=4.1466


,seed,enriched,enriched_cold_start
0,1,0.031148,0.024590
1,7,0.031148,0.022951
2,42,0.022951,0.018033


Across every seed I tried, `enriched_cold_start` is **never worse** than
`enriched`, and is sometimes noticeably better. That rules out "this was a
one-off noisy run" — it's a reproducible pattern, not an artifact.

The honest explanation, given how this model is built: with only ~610 users
and a median of well under 100 ratings each, there just isn't much
per-user signal for a 16-dim identity embedding to learn beyond what the
`user_take` (recent-history) feature and shared item/context embeddings
already capture — and `user_take` gets gradient updates from *every* user who
ever rated a given movie, not just from one user's own sparse history, so it
trains far more thoroughly. On this dataset, at this scale, the architecture
doesn't seem to need per-user identity at all once recency + context are
present. That's not a failure of the cold-start mechanism — if anything, it's
evidence *for* it: the feature that makes cold-start work (`user_take`) turns
out to be carrying most of the useful signal even for returning users, not
just new ones. Whether that holds at Joyn's actual scale (millions of users,
far more per-user history) is a genuinely different question — plausibly not,
since there'd be enough per-user data for an identity embedding to earn its
keep. Worth stating as a scale-dependent finding, not a universal one, in
ADR-001/ADR-004.


## Export serving artifacts

In [10]:
@torch.no_grad()
def export_artifacts(model: TwoTower, eval_results: dict):
    item_embs = all_item_embeddings(model).numpy().astype("float32")
    idx_to_movie_id = {v: int(k) for k, v in movie_vocab.items()}
    movie_ids_ordered = np.array([idx_to_movie_id.get(i, -1) for i in range(n_movies)])

    np.savez(
        ARTIFACTS_DIR / "item_embeddings.npz",
        movie_ids=movie_ids_ordered,
        embeddings=item_embs,
    )

    sd = model.query_tower.state_dict()
    np.savez(
        ARTIFACTS_DIR / "query_tower_weights.npz",
        user_embedding=sd["user.weight"].numpy(),
        movie_embedding=model.shared.movie.weight.detach().numpy(),
        day_embedding=sd["day.weight"].numpy(),
        tod_embedding=sd["tod.weight"].numpy(),
        mlp_0_weight=sd["mlp.0.weight"].numpy(),
        mlp_0_bias=sd["mlp.0.bias"].numpy(),
        mlp_2_weight=sd["mlp.2.weight"].numpy(),
        mlp_2_bias=sd["mlp.2.bias"].numpy(),
    )

    with open(ARTIFACTS_DIR / "feature_vocab.json", "w") as f:
        json.dump(
            {
                "user_vocab": vocab["user_vocab"],
                "movie_vocab": movie_vocab,
                "day_vocab": day_vocab,
                "tod_vocab": tod_vocab,
                "user_take_k": K,
                "embedding_dims": {
                    "user": EMB_USER, "movie": EMB_MOVIE, "day": EMB_DAY, "tod": EMB_TOD,
                    "tower_out": TOWER_OUT_DIM,
                },
            },
            f,
        )

    with open(ARTIFACTS_DIR / "eval_metrics.json", "w") as f:
        json.dump(eval_results, f, indent=2)

    print("wrote artifacts/item_embeddings.npz, query_tower_weights.npz, "
          "feature_vocab.json, eval_metrics.json")


export_artifacts(enriched_model, results)


wrote artifacts/item_embeddings.npz, query_tower_weights.npz, feature_vocab.json, eval_metrics.json


**Weight export note:** `movie_embedding` comes from `model.shared` (it's the
one table both towers touch — the item tower embeds the current movie with
it, the query tower mean-pools it for `user_take`). `user_embedding`,
`day_embedding`, and `tod_embedding` live on `query_tower` only. Nothing from
the item tower's genre/text path is exported — the backend never runs the
item tower at serving time, only `item_embeddings.npz` (already computed for
the whole catalog above).


## My take — ADR-001 and ADR-004

Both of these now have real numbers to react to (the `results` table above):

- **ADR-001** (two-tower vs. matrix factorization, at this scale): does the
  `enriched` model's overall Recall@10/NDCG@10 justify the extra
  machinery over `plain`, for a catalog this size? Or is the real win
  somewhere the aggregate metric doesn't show well (see below)?
- **ADR-004** (the cold-start trick): how much recall does
  `enriched_cold_start` lose relative to `enriched`'s normal-user numbers?
  Is that gap small enough that you'd call the cold-start mechanism a
  success, or large enough that it's "better than nothing" but not a real
  solution?

Write both into `docs/ARCHITECTURE_DECISIONS.md` using the actual numbers
printed above — not a generic "yes context helps" statement.
